In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/raw/test.csv')

print(f"✅ Raw data loaded")
print(f"Shape before cleaning: {df.shape[0]:,} rows, {df.shape[1]} columns")

✅ Raw data loaded
Shape before cleaning: 625,134 rows, 9 columns


In [2]:
# Convert from text to real datetime format
df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'])

print("✅ pickup_datetime converted to datetime")
print(f"Data type now: {df['pickup_datetime'].dtype}")
print(f"Sample: {df['pickup_datetime'].head(3).tolist()}")

✅ pickup_datetime converted to datetime
Data type now: datetime64[ns]
Sample: [Timestamp('2016-06-30 23:59:00'), Timestamp('2016-06-30 23:59:00'), Timestamp('2016-06-30 23:59:00')]


In [3]:
# Official NYC boundaries from NYC.gov
LAT_MIN, LAT_MAX = 40.496, 40.916
LON_MIN, LON_MAX = -74.269, -73.699

rows_before = len(df)

df = df[
    (df['pickup_latitude']  >= LAT_MIN) & (df['pickup_latitude']  <= LAT_MAX) &
    (df['pickup_longitude'] >= LON_MIN) & (df['pickup_longitude'] <= LON_MAX) &
    (df['dropoff_latitude']  >= LAT_MIN) & (df['dropoff_latitude']  <= LAT_MAX) &
    (df['dropoff_longitude'] >= LON_MIN) & (df['dropoff_longitude'] <= LON_MAX)
]

rows_after = len(df)
removed = rows_before - rows_after

print(f"✅ GPS filter applied")
print(f"Rows before : {rows_before:,}")
print(f"Rows after  : {rows_after:,}")
print(f"Rows removed: {removed:,} ({removed/rows_before*100:.2f}%)")

✅ GPS filter applied
Rows before : 625,134
Rows after  : 624,612
Rows removed: 522 (0.08%)


In [4]:
rows_before = len(df)

# Keep only trips with 1 to 5 passengers (NYC TLC official maximum)
# Source: NYC.gov TLC - nyc.gov/site/tlc/passengers
df = df[(df['passenger_count'] >= 1) & (df['passenger_count'] <= 5)]

rows_after = len(df)
removed = rows_before - rows_after

print(f"✅ Passenger count filter applied")
print(f"Rows before : {rows_before:,}")
print(f"Rows after  : {rows_after:,}")
print(f"Rows removed: {removed:,}")
print(f"\nPassenger count values remaining:")
print(df['passenger_count'].value_counts().sort_index())

✅ Passenger count filter applied
Rows before : 624,612
Rows after  : 604,075
Rows removed: 20,537

Passenger count values remaining:
passenger_count
1    443033
2     89970
3     25668
4     12010
5     33394
Name: count, dtype: int64


In [5]:
# Convert Y/N text to 1/0 integer
df['store_and_fwd_flag'] = df['store_and_fwd_flag'].map({'N': 0, 'Y': 1})

print("✅ store_and_fwd_flag converted")
print(df['store_and_fwd_flag'].value_counts())

✅ store_and_fwd_flag converted
store_and_fwd_flag
0    600655
1      3420
Name: count, dtype: int64


In [6]:
# All column names are already snake_case — just verify they're clean
print("✅ Column names check:")
print(df.columns.tolist())

# Rename pickup_datetime to be more explicit
df = df.rename(columns={'pickup_datetime': 'pickup_datetime'})  

print("\nAll column names are clean and coding-friendly ✅")

✅ Column names check:
['id', 'vendor_id', 'pickup_datetime', 'passenger_count', 'pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude', 'store_and_fwd_flag']

All column names are clean and coding-friendly ✅


In [7]:
print("=== FINAL CLEANED DATASET SUMMARY ===")
print(f"Total rows    : {len(df):,}")
print(f"Total columns : {df.shape[1]}")

print("\n=== DATA TYPES ===")
print(df.dtypes)

print("\n=== MISSING VALUES ===")
print(df.isnull().sum())

print("\n=== GPS RANGES (should be within NYC) ===")
print(f"Pickup Longitude  : {df['pickup_longitude'].min():.4f} to {df['pickup_longitude'].max():.4f}")
print(f"Pickup Latitude   : {df['pickup_latitude'].min():.4f} to {df['pickup_latitude'].max():.4f}")

print("\n=== PASSENGER COUNT RANGE ===")
print(f"Min: {df['passenger_count'].min()} | Max: {df['passenger_count'].max()}")

print("\n=== STORE AND FWD FLAG VALUES ===")
print(df['store_and_fwd_flag'].value_counts())

=== FINAL CLEANED DATASET SUMMARY ===
Total rows    : 604,075
Total columns : 9

=== DATA TYPES ===
id                            object
vendor_id                      int64
pickup_datetime       datetime64[ns]
passenger_count                int64
pickup_longitude             float64
pickup_latitude              float64
dropoff_longitude            float64
dropoff_latitude             float64
store_and_fwd_flag             int64
dtype: object

=== MISSING VALUES ===
id                    0
vendor_id             0
pickup_datetime       0
passenger_count       0
pickup_longitude      0
pickup_latitude       0
dropoff_longitude     0
dropoff_latitude      0
store_and_fwd_flag    0
dtype: int64

=== GPS RANGES (should be within NYC) ===
Pickup Longitude  : -74.2683 to -73.7041
Pickup Latitude   : 40.5187 to 40.9130

=== PASSENGER COUNT RANGE ===
Min: 1 | Max: 5

=== STORE AND FWD FLAG VALUES ===
store_and_fwd_flag
0    600655
1      3420
Name: count, dtype: int64


In [8]:
# Save to data/cleaned/
output_path = '../data/cleaned/taxi_cleaned.csv'
df.to_csv(output_path, index=False)

print(f"✅ Cleaned file saved to: {output_path}")
print(f"Final shape: {df.shape[0]:,} rows, {df.shape[1]} columns")

✅ Cleaned file saved to: ../data/cleaned/taxi_cleaned.csv
Final shape: 604,075 rows, 9 columns
